# Elo overview

Exploration rapide des Elo (global et par patch) générés dans `data/metrics/`.
Plots allégés pour rester lisibles (top restreint, rang hebdo) et table leaderboard.


In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
import plotly.express as px

def find_root() -> Path:
    cand = Path.cwd()
    for c in [cand, *cand.parents]:
        if (c / "data" / "metrics").exists():
            return c
    return cand

ROOT = find_root()
METRICS_DIR = ROOT / "data" / "metrics"
TS_PATH = METRICS_DIR / "elo_timeseries.parquet"
LATEST_PATH = METRICS_DIR / "elo_latest.parquet"
PATCH_LATEST_PATH = METRICS_DIR / "elo_patch_latest.parquet"
TRACKED_PATH = METRICS_DIR / "tracked_teams.parquet"

def load_df(path: Path) -> pl.DataFrame | None:
    if path.exists():
        return pl.read_parquet(path)
    print(f"Missing: {path}")
    return None

elo_ts = load_df(TS_PATH)
elo_latest = load_df(LATEST_PATH)
elo_patch_latest = load_df(PATCH_LATEST_PATH)
tracked = load_df(TRACKED_PATH)

names_map = {}
if tracked is not None and not tracked.is_empty():
    for r in tracked.iter_rows(named=True):
        tid = r.get("team_id") or r.get("TeamID")
        nm = r.get("name") or r.get("TeamName")
        if tid is not None and nm:
            names_map[int(tid)] = nm

def add_names(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or not names_map:
        return df
    df = df.copy()
    df["team_name"] = df["team_id"].map(names_map)
    df["team_label"] = df.apply(
        lambda r: f"{r['team_name']} ({r['team_id']})" if pd.notnull(r.get('team_name')) else r['team_id'], axis=1
    )
    return df

elo_ts_pandas = add_names(elo_ts.to_pandas()) if elo_ts is not None and not elo_ts.is_empty() else pd.DataFrame()
elo_latest_pandas = add_names(elo_latest.to_pandas()) if elo_latest is not None and not elo_latest.is_empty() else pd.DataFrame()
elo_patch_latest_pandas = add_names(elo_patch_latest.to_pandas()) if elo_patch_latest is not None and not elo_patch_latest.is_empty() else pd.DataFrame()
elo_ts_pandas.head()


,match_id,start_time,start_dt,team_id,opponent_id,team_is_radiant,team_win,rating_pre,rating_post,rating_pre_patch,rating_post_patch,expected,expected_patch,tournament_tier,tournament_location,weight,patch,team_name,team_label
0,7520706781,1704355281,2024-01-04 08:01:21,8588969,9081007,False,0,1500.000000,1480.207524,1500.000000,1480.207524,0.471249,0.471249,qualifier,online,1.05,54,HYDRA,HYDRA (8588969)
1,7520706865,1704355380,2024-01-04 08:03:00,2576071,9279613,False,1,1500.000000,1522.207524,1500.000000,1522.207524,0.471249,0.471249,qualifier,online,1.05,54,Yellow Submarine,Yellow Submarine (2576071)
2,7520754303,1704358792,2024-01-04 08:59:52,8588969,9081007,False,0,1480.207524,1462.774110,1480.207524,1462.774110,0.415081,0.415081,qualifier,online,1.05,54,HYDRA,HYDRA (8588969)
3,7520761790,1704358994,2024-01-04 09:03:14,2576071,9279613,True,1,1522.207524,1539.358100,1522.207524,1539.358100,0.591653,0.591653,qualifier,online,1.05,54,Yellow Submarine,Yellow Submarine (2576071)
4,7523989094,1704528110,2024-01-06 08:01:50,8376696,9216247,True,1,1500.000000,1519.792476,1500.000000,1519.792476,0.528751,0.528751,qualifier,online,1.05,54,One Move,One Move (8376696)


## Top 20 actuel (global)
Affiche les 20 meilleurs Elo globaux (avec labels si disponibles).


In [3]:
if elo_latest_pandas.empty:
    print("No elo_latest available")
else:
    top20 = elo_latest_pandas.sort_values("elo", ascending=False).head(10)
    y_col = "team_label" if "team_label" in top20.columns else "team_id"
    fig = px.bar(top20, x="elo", y=y_col, orientation="h", title="Top 10 Elo (global)", text="elo")
    fig.update_layout(yaxis=dict(autorange="reversed"))
    fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
    fig.show()
    display(top20[["team_id", "team_name", "elo"]])

,team_id,team_name,elo
0,9247354,Team Falcons,1955.748845
1,7119388,Team Spirit,1915.597809
2,9572001,PVISION,1900.978797
3,8291895,Tundra Esports,1890.674325
4,9338413,MOUZ,1861.609263
5,2163,Team Liquid,1827.933511
6,9823272,Team Yandex,1820.878223
7,8255888,BB Team,1818.308657
8,9766941,FLIPSTER TALON,1783.003350
9,9634742,Chimera Esports,1778.915298


In [2]:
if elo_latest_pandas.empty:
    print("No elo_latest available")
else:
    top20 = elo_latest_pandas.sort_values("elo", ascending=False).head(20)
    y_col = "team_label" if "team_label" in top20.columns else "team_id"
    fig = px.bar(top20, x="elo", y=y_col, orientation="h", title="Top 20 Elo (global)", text="elo")
    fig.update_layout(yaxis=dict(autorange="reversed"))
    fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
    fig.show()
    display(top20[["team_id", "team_name", "elo"]])


,team_id,team_name,elo
0,9247354,Team Falcons,1955.748845
1,7119388,Team Spirit,1915.597809
2,9572001,PVISION,1900.978797
3,8291895,Tundra Esports,1890.674325
4,9338413,MOUZ,1861.609263
5,2163,Team Liquid,1827.933511
6,9823272,Team Yandex,1820.878223
7,8255888,BB Team,1818.308657
8,9766941,FLIPSTER TALON,1783.003350
9,9634742,Chimera Esports,1778.915298


## Évolution du top 10 (rank hebdo)

Dernier Elo par équipe et par semaine; rang hebdo; on trace les équipes qui sont au moins une fois dans le top 10.


In [4]:
if elo_ts_pandas.empty:
    print("No elo_timeseries available")
else:
    ts = elo_ts_pandas.copy()
    ts["start_dt"] = pd.to_datetime(ts["start_time"], unit="s")
    ts["week"] = ts["start_dt"].dt.to_period("W").apply(lambda r: r.start_time)
    ts_weekly = ts.sort_values(["team_id", "start_dt"]).groupby(["team_id", "week"]).tail(1)
    ts_weekly["rank"] = ts_weekly.groupby("week")["rating_post"].rank(ascending=False, method="dense")
    ts_top = ts_weekly[ts_weekly["rank"] <= 10]
    y_col = "team_label" if "team_label" in ts_top.columns else "team_id"
    fig = px.line(
        ts_top,
        x="week",
        y="rank",
        color=y_col,
        hover_data=["rating_post"],
        title="Évolution du top 10 Elo (global, hebdo)",
    )
    fig.update_yaxes(autorange="reversed")
    fig.show()
    display(ts_top[["week", "team_id", "team_name", "rank", "rating_post"]].head())


,week,team_id,team_name,rank,rating_post
1697,2024-03-18,36,Natus Vincere,10.0,1595.646200
3119,2024-05-20,36,Natus Vincere,10.0,1558.304600
4882,2024-07-29,36,Natus Vincere,10.0,1600.319432
5181,2024-08-12,36,Natus Vincere,9.0,1595.780883
6475,2024-10-07,36,Natus Vincere,5.0,1647.343930


## Top Elo par patch (top 10 par patch)


In [5]:
if elo_patch_latest_pandas.empty:
    print("No elo_patch_latest available")
else:
    top_patch = (
        elo_patch_latest_pandas.sort_values(["patch", "elo"], ascending=[True, False])
        .groupby("patch")
        .head(10)
    )
    y_col = "team_label" if "team_label" in top_patch.columns else "team_id"
    fig = px.bar(
        top_patch,
        x="elo",
        y=y_col,
        color="patch",
        orientation="h",
        title="Top 10 Elo par patch",
    )
    fig.update_layout(yaxis=dict(autorange="reversed"))
    fig.show()
    display(top_patch[["patch", "team_id", "team_name", "elo"]])


,patch,team_id,team_name,elo
0,54,9247354,Team Falcons,1832.503880
1,54,8261500,Xtreme Gaming,1793.329358
2,54,7119388,Team Spirit,1728.558514
3,54,8291895,Tundra Esports,1710.130644
4,54,8599101,Gaimin Gladiators,1695.502974
5,54,2586976,OG,1668.830376
6,54,9017006,NAVI Junior,1647.018705
7,54,8255888,BB Team,1646.176158
8,54,9338413,MOUZ,1644.095411
9,54,2163,Team Liquid,1633.236693


## Leaderboard complet (global)


In [6]:
if elo_latest_pandas.empty:
    print("No elo_latest available")
else:
    leaderboard = elo_latest_pandas.copy()
    leaderboard["elo_rank"] = leaderboard["elo"].rank(ascending=False, method="dense").astype(int)
    leaderboard = leaderboard.sort_values("elo_rank").reset_index(drop=True)
    cols = [c for c in ["elo_rank", "team_id", "team_name", "elo"] if c in leaderboard.columns]
    display(leaderboard[cols].head(50))


,elo_rank,team_id,team_name,elo
0,1,9247354,Team Falcons,1955.748845
1,2,7119388,Team Spirit,1915.597809
2,3,9572001,PVISION,1900.978797
3,4,8291895,Tundra Esports,1890.674325
4,5,9338413,MOUZ,1861.609263
5,6,2163,Team Liquid,1827.933511
6,7,9823272,Team Yandex,1820.878223
7,8,8255888,BB Team,1818.308657
8,9,9766941,FLIPSTER TALON,1783.003350
9,10,9634742,Chimera Esports,1778.915298
